In [1]:
import ssl
import socket
import re
import math
from datetime import datetime, timezone
from urllib.parse import urlparse
from difflib import SequenceMatcher

import joblib
import pandas as pd
import requests
import whois
import tldextract

# -------------------------------------------------------------
# 1. LOAD THE TRAINED MODEL
# -------------------------------------------------------------
model_filename = 'phishing_rf_model.pkl'
try:
    model = joblib.load(model_filename)
    expected_features = list(model.feature_names_in_)
    print(f"[*] Loaded '{model_filename}' successfully ({len(expected_features)} features expected).")
except Exception as e:
    print(f"[!] Failed to load model: {e}")
    raise SystemExit

# -------------------------------------------------------------
# 2. HELPER FUNCTIONS FOR LIVE FEATURE EXTRACTION
# -------------------------------------------------------------
POPULAR_BRANDS = [
    'google', 'microsoft', 'apple', 'amazon', 'paypal', 
    'facebook', 'netflix', 'instagram', 'whatsapp', 'yahoo', 
    'chase', 'wellsfargo', 'bankofamerica', 'gcash', 'maya'
]

SUSPICIOUS_TLDS = {
    'tk', 'ml', 'ga', 'cf', 'gq', 'top', 'xyz', 'work', 
    'buzz', 'icu', 'fit', 'club', 'surf', 'live'
}

SUSPICIOUS_KEYWORDS = [
    'login', 'signin', 'verify', 'verification', 'update', 
    'secure', 'account', 'banking', 'confirm', 'password', 
    'wallet', 'authenticate', 'service', 'support'
]

TRUSTED_ISSUERS = [
    "DigiCert", "Let's Encrypt", "GlobalSign", "Sectigo", 
    "Google Trust Services", "Amazon", "GoDaddy", "Cloudflare"
]

def calculate_shannon_entropy(text):
    if not text:
        return 0.0
    prob_dict = {c: text.count(c) / len(text) for c in set(text)}
    return -sum(p * math.log2(p) for p in prob_dict.values())

def get_levenshtein_and_similarity(domain_name):
    clean_domain = domain_name.lower().split('.')[0]
    best_dist = 999
    best_ratio = 0.0
    for brand in POPULAR_BRANDS:
        ratio = SequenceMatcher(None, clean_domain, brand).ratio()
        if ratio > best_ratio:
            best_ratio = ratio
        # Simple Levenshtein distance approximation via length & edit differences
        dist = abs(len(clean_domain) - len(brand)) + sum(1 for a, b in zip(clean_domain, brand) if a != b)
        if dist < best_dist:
            best_dist = dist
    return float(best_dist), float(best_ratio)

def extract_whois_features(domain):
    try:
        w = whois.whois(domain)
        creation = w.creation_date
        expiration = w.expiration_date
        
        # Normalize list outputs from WHOIS responses
        if isinstance(creation, list):
            creation = creation[0]
        if isinstance(expiration, list):
            expiration = expiration[0]
            
        now = datetime.now()
        age_days = (now - creation).days if creation and isinstance(creation, datetime) else -1
        expiry_days = (expiration - now).days if expiration and isinstance(expiration, datetime) else -1
        
        raw_text = str(w).lower()
        whois_priv = 1 if any(k in raw_text for k in ['privacy', 'redact', 'whoisguard', 'proxy', 'withheld']) else 0
        reg_recently = 1 if 0 <= age_days < 90 else 0
        
        return age_days, expiry_days, whois_priv, reg_recently
    except Exception:
        return -1, -1, -1, -1

def extract_ssl_features(hostname):
    ctx = ssl.create_default_context()
    try:
        with socket.create_connection((hostname, 443), timeout=4) as sock:
            with ctx.wrap_socket(sock, server_hostname=hostname) as ssock:
                cert = ssock.getpeercert()
                
                not_before = datetime.strptime(cert['notBefore'], '%b %d %H:%M:%S %Y %Z')
                not_after = datetime.strptime(cert['notAfter'], '%b %d %H:%M:%S %Y %Z')
                now = datetime.now(timezone.utc).replace(tzinfo=None)
                
                age_days = (now - not_before).days
                is_expired = 1 if now > not_after else 0
                
                issuer_str = str(cert.get('issuer', ''))
                issuer_trusted = 1 if any(t.lower() in issuer_str.lower() for t in TRUSTED_ISSUERS) else 0
                
                # Check self-signed
                subject_str = str(cert.get('subject', ''))
                self_signed = 1 if issuer_str == subject_str else 0
                
                return 1, is_expired, self_signed, age_days, issuer_trusted
    except ssl.SSLCertVerificationError:
        return 0, 0, 1, -1, 0
    except Exception:
        # Connection timeout, port 443 closed, or plain HTTP
        return -1, -1, -1, -1, -1

def extract_network_redirect_features(url, parsed_domain):
    try:
        session = requests.Session()
        resp = session.get(url, timeout=5, allow_redirects=True, headers={'User-Agent': 'Mozilla/5.0'})
        
        history = resp.history
        redirect_count = len(history)
        
        domains_chain = [urlparse(h.url).netloc for h in history] + [urlparse(resp.url).netloc]
        unique_domains = len(set(domains_chain))
        cross_domain = sum(1 for i in range(len(domains_chain) - 1) if domains_chain[i] != domains_chain[i+1])
        
        final_host = urlparse(resp.url).netloc
        final_ext = tldextract.extract(resp.url)
        orig_ext = tldextract.extract(url)
        same_reg = 1 if final_ext.top_domain_under_public_suffix == orig_ext.top_domain_under_public_suffix else 0
        
        final_https = 1 if resp.url.lower().startswith('https://') else 0
        redirect_to_ip = 1 if re.match(r'^\d{1,3}(\.\d{1,3}){3}$', final_host) else 0
        
        return redirect_count, cross_domain, unique_domains, same_reg, final_https, redirect_to_ip
    except Exception:
        return 0, 0, 1, 1, (1 if url.startswith('https') else 0), 0

# -------------------------------------------------------------
# 3. CONSOLIDATED 38-FEATURE LIVE EXTRACTOR
# -------------------------------------------------------------
def extract_live_features(raw_url):
    url = raw_url.strip()
    if not url.startswith(('http://', 'https://')):
        url = 'http://' + url

    parsed = urlparse(url)
    hostname = parsed.netloc.split(':')[0]
    path = parsed.path
    query = parsed.query
    ext = tldextract.extract(url)

    # 1. Lexical features
    url_len = len(url)
    host_len = len(hostname)
    path_len = len(path)
    query_len = len(query)
    num_digits = sum(c.isdigit() for c in url)
    
    lexical = {
        'url_length': url_len,
        'hostname_length': host_len,
        'path_length': path_len,
        'query_length': query_len,
        'num_dots': url.count('.'),
        'num_hyphens': url.count('-'),
        'num_underscores': url.count('_'),
        'num_slashes': url.count('/'),
        'num_questionmarks': url.count('?'),
        'num_equals': url.count('='),
        'num_at': url.count('@'),
        'num_digits': num_digits,
        'num_subdomains': len(ext.subdomain.split('.')) if ext.subdomain else 0,
        'digit_ratio': (num_digits / url_len) if url_len > 0 else 0.0,
        'special_char_ratio': (sum(not c.isalnum() for c in url) / url_len) if url_len > 0 else 0.0,
        'url_entropy': calculate_shannon_entropy(url),
        'has_ip_address': 1 if re.match(r'^\d{1,3}(\.\d{1,3}){3}$', hostname) else 0,
        'has_https': 1 if url.startswith('https://') else 0,
        'suspicious_keyword_count': sum(1 for kw in SUSPICIOUS_KEYWORDS if kw in url.lower()),
        'suspicious_tld': 1 if ext.suffix.lower() in SUSPICIOUS_TLDS else 0,
    }

    # 2. Similarity features
    lev_dist, sim_score = get_levenshtein_and_similarity(ext.domain)
    lexical['levenshtein_distance'] = lev_dist
    lexical['brand_similarity'] = sim_score

    # 3. WHOIS features
    age, expiry, privacy, reg_recent = extract_whois_features(ext.top_domain_under_public_suffix or hostname)
    lexical['domain_age_days'] = age
    lexical['domain_expiry_days'] = expiry
    lexical['whois_privacy'] = privacy
    lexical['domain_registered_recently'] = reg_recent

    # 4. SSL features
    ssl_v, ssl_exp, ssl_self, cert_age, cert_trusted = extract_ssl_features(hostname)
    lexical['ssl_valid'] = ssl_v
    lexical['ssl_expired'] = ssl_exp
    lexical['ssl_self_signed'] = ssl_self
    lexical['certificate_age_days'] = cert_age
    lexical['certificate_issuer_trusted'] = cert_trusted

    # 5. Redirects and Network features
    red_cnt, cross_red, uniq_dom, same_dom, fin_https, red_ip = extract_network_redirect_features(url, hostname)
    lexical['redirect_count'] = red_cnt
    lexical['cross_domain_redirects'] = cross_red
    lexical['unique_domains_in_chain'] = uniq_dom
    lexical['same_registered_domain'] = same_dom
    lexical['final_https'] = fin_https
    lexical['redirect_to_ip'] = red_ip

    # 6. Popularity rank
    # Defaults to -1 if no local Tranco lookup list is loaded
    lexical['tranco_rank'] = -1

    # Map strictly into expected 38-feature order
    row_values = [lexical.get(col, 0) for col in expected_features]
    return pd.DataFrame([row_values], columns=expected_features)

# -------------------------------------------------------------
# 4. TEST PIPELINE
# -------------------------------------------------------------
# Change this URL to test any link live
TEST_URL = "https://www.google.com"

print("=" * 60)
print(f"Analyzing Live URL: {TEST_URL}")
print("Gathering Lexical, WHOIS, SSL, and Redirect Metrics...")
print("=" * 60)

live_features_df = extract_live_features(TEST_URL)

prediction = model.predict(live_features_df)[0]
probabilities = model.predict_proba(live_features_df)[0]
confidence = probabilities[prediction] * 100

verdict = "🚩 PHISHING" if prediction == 1 else "✅ SAFE"
print(f"\nFinal Verdict : {verdict}")
print(f"Confidence    : {confidence:.2f}%\n")

# Top influential features breakdown
print(f"{'Feature Name':<30} | {'Global Weight'} | {'Live Extracted Value'}")
print("-" * 65)
breakdown = pd.DataFrame({
    'Feature': expected_features,
    'Weight': model.feature_importances_,
    'Value': live_features_df.iloc[0].values
}).sort_values(by='Weight', ascending=False)

for _, row in breakdown.head(10).iterrows():
    print(f"{row['Feature']:<30} | {row['Weight']:.4f}        | {row['Value']}")

c:\Users\Strix\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.8.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Strix\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.8.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


[*] Loaded 'phishing_rf_model.pkl' successfully (38 features expected).
Analyzing Live URL: https://www.google.com
Gathering Lexical, WHOIS, SSL, and Redirect Metrics...

Final Verdict : ✅ SAFE
Confidence    : 84.95%

Feature Name                   | Global Weight | Live Extracted Value
-----------------------------------------------------------------
final_https                    | 0.1276        | 1.0
same_registered_domain         | 0.1074        | 1.0
redirect_to_ip                 | 0.0953        | 0.0
unique_domains_in_chain        | 0.0894        | 1.0
cross_domain_redirects         | 0.0829        | 0.0
redirect_count                 | 0.0519        | 0.0
url_entropy                    | 0.0407        | 3.6635327548042547
certificate_issuer_trusted     | 0.0405        | 1.0
url_length                     | 0.0382        | 22.0
certificate_age_days           | 0.0377        | 21.0
